# Scaffold-Based Train, Validation and Test Split

This notebook creates scaffold-based training, validation and test sets using Bemis–Murcko molecular scaffolds. Molecules sharing the same scaffold are kept within the same split to reduce structural leakage, while scaffold-level activity groups are used to maintain a similar class distribution across the three sets. The final split contains separate training, validation and held-out scaffold test sets with zero scaffold overlap.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

DATA_FILE = "preprocessed_model_data.csv"

CLASSIFICATION_RESULTS_FILE = (
    "classification_validation_results.csv"
)

CLASSIFICATION_PREDICTIONS_FILE = (
    "classification_validation_predictions.csv"
)

REGRESSION_PREDICTIONS_FILE = (
    "regression_validation_predictions.csv"
)

DOCKING_CUTOFF = -8.0


data_df = pd.read_csv(DATA_FILE)

classification_results_df = pd.read_csv(
    CLASSIFICATION_RESULTS_FILE
)

classification_predictions_df = pd.read_csv(
    CLASSIFICATION_PREDICTIONS_FILE
)

regression_predictions_df = pd.read_csv(
    REGRESSION_PREDICTIONS_FILE
)


validation_labels_df = data_df.loc[
    data_df["split"] == "validation",
    ["canonical_smiles", "dual_candidate"]
].copy()


print("Validation molecules:", len(validation_labels_df))

print(
    "Classification prediction rows:",
    len(classification_predictions_df)
)

print(
    "Regression prediction rows:",
    len(regression_predictions_df)
)

print("\nThe test set has not been used.")

In [ ]:
# Create indirect dual-candidate predictions

# Place the GSK3B and JNK3 predictions in separate columns
indirect_predictions_df = (
    regression_predictions_df
    .pivot(
        index=["canonical_smiles", "model"],
        columns="target",
        values="predicted_score"
    )
    .reset_index()
)

# Rename the score columns clearly
indirect_predictions_df = indirect_predictions_df.rename(
    columns={
        "GSK3B": "predicted_gsk3b_score",
        "JNK3": "predicted_jnk3_score"
    }
)

# Add the true dual-candidate labels
indirect_predictions_df = indirect_predictions_df.merge(
    validation_labels_df,
    on="canonical_smiles",
    how="left"
)

# A molecule passes only when BOTH predicted scores are <= -8
indirect_predictions_df["predicted_dual_candidate"] = (
    (
        indirect_predictions_df["predicted_gsk3b_score"]
        <= DOCKING_CUTOFF
    )
    &
    (
        indirect_predictions_df["predicted_jnk3_score"]
        <= DOCKING_CUTOFF
    )
).astype(int)

# The weaker predicted target score
indirect_predictions_df["bottleneck_score"] = (
    indirect_predictions_df[
        [
            "predicted_gsk3b_score",
            "predicted_jnk3_score"
        ]
    ]
    .max(axis=1)
)

print("Indirect prediction rows:", len(indirect_predictions_df))

print("\nRows for each regression model:")
print(indirect_predictions_df["model"].value_counts())

In [ ]:
#  Evaluate regression-derived classification

def calculate_classification_metrics(
    model_name,
    true_labels,
    predicted_labels,
    ranking_scores
):

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        predicted_labels
    ).ravel()

    specificity = tn / (tn + fp)

    return {
        "method": "Regression-derived",
        "model": model_name,

        "balanced_accuracy": balanced_accuracy_score(
            true_labels,
            predicted_labels
        ),

        "mcc": matthews_corrcoef(
            true_labels,
            predicted_labels
        ),

        "roc_auc": roc_auc_score(
            true_labels,
            ranking_scores
        ),

        "average_precision": average_precision_score(
            true_labels,
            ranking_scores
        ),

        "precision": precision_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "recall": recall_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "specificity": specificity,

        "f1_score": f1_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp
    }


indirect_results = []


for model_name in indirect_predictions_df["model"].unique():

    model_predictions = indirect_predictions_df[
        indirect_predictions_df["model"] == model_name
    ]

    true_labels = model_predictions[
        "dual_candidate"
    ]

    predicted_labels = model_predictions[
        "predicted_dual_candidate"
    ]

    # More negative bottleneck scores indicate stronger predictions.
    # Multiplying by -1 makes higher values represent stronger candidates.
    ranking_scores = -model_predictions[
        "bottleneck_score"
    ]

    result = calculate_classification_metrics(
        model_name,
        true_labels,
        predicted_labels,
        ranking_scores
    )

    indirect_results.append(result)


indirect_results_df = pd.DataFrame(
    indirect_results
)

display(
    indirect_results_df.round(3)
)

In [ ]:
#  Compare direct classifiers with regression-derived classifiers

direct_results_df = classification_results_df.copy()

direct_results_df.insert(
    0,
    "method",
    "Direct classification"
)

comparison_columns = [
    "method",
    "model",
    "balanced_accuracy",
    "mcc",
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "specificity",
    "f1_score",
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive"
]

comparison_results_df = pd.concat(
    [
        direct_results_df[comparison_columns],
        indirect_results_df[comparison_columns]
    ],
    ignore_index=True
)

comparison_results_df = comparison_results_df.sort_values(
    by="balanced_accuracy",
    ascending=False
)

display(
    comparison_results_df.round(3)
)

In [ ]:
# Save comparison results and indirect predictions

comparison_results_df.to_csv(
    "direct_vs_indirect_validation_results.csv",
    index=False
)

indirect_predictions_df.to_csv(
    "indirect_validation_predictions.csv",
    index=False
)


best_direct = (
    comparison_results_df[
        comparison_results_df["method"]
        == "Direct classification"
    ]
    .sort_values(
        by="balanced_accuracy",
        ascending=False
    )
    .iloc[0]
)

best_indirect = (
    comparison_results_df[
        comparison_results_df["method"]
        == "Regression-derived"
    ]
    .sort_values(
        by="balanced_accuracy",
        ascending=False
    )
    .iloc[0]
)


print("Best direct classification model:")
print(best_direct[["model", "balanced_accuracy", "mcc"]])

print("\nBest regression-derived model:")
print(best_indirect[["model", "balanced_accuracy", "mcc"]])

print("\nSaved: direct_vs_indirect_validation_results.csv")
print("Saved: indirect_validation_predictions.csv")